# Why short exposure is invisible to the audit, and what sees it

Reviewer 1, comment 3, and Reviewer 2, comment 4, of the SoftwareX submission. This notebook
gives the numbers behind the short-exposure passage of Section 3.2, the accuracy-cost column
of Table 2 (formerly Table 1) and the corresponding part of `docs/degradation.md`.

**Requirements.** `pypasi >= 0.5.5` and the Ho et al. `.npy` files in `Bacteria/`. Runtime is a few minutes.

## 0 · Setup (shared with `parameter_guide.ipynb`), and the paper's model on split 0

In [1]:
# --- shared setup: the data, the patient-disjoint split and the defects of the paper ---
from pathlib import Path
import time, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
import pypasi
from pypasi import BandNegotiationClassifier, BandSet, ControlProfile, PLSDA, Triage, degrade
from pypasi.datasets import load_bacteria

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Control limits built from a random split")
assert tuple(int(p) for p in pypasi.__version__.split(".")[:3]) >= (0, 5, 5), "needs pypasi >= 0.5.5"

DATA_DIR = Path("Bacteria")
SEED = 0
# the configuration of softwarex_examples.ipynb; every function below defaults to it
N_BANDS, TOPOLOGY, REGIME, N_COMPONENTS, SCALE_NAME = 7, "complete", "H1", 10, "SNV"
CALIBRATION, CAL_SPLIT = "auto", 0.25
REVIEW_RATE, NOVELTY_Q = 0.10, 0.99
K_SIGMA, LASER_NM, PROFILE_STATS = 3.0, 633.0, ("mean_stress", "band_confidence")
BATCH_N, LOCALISE_REPEATS = 100, 6
SPECTRA_PER_PATIENT = 400


def minmax(X):
    X = np.asarray(X, float)
    lo, hi = X.min(1, keepdims=True), X.max(1, keepdims=True)
    return (X - lo) / np.where(hi > lo, hi - lo, 1.0)


def snv(X):
    X = np.asarray(X, float)
    s = X.std(1, keepdims=True)
    return (X - X.mean(1, keepdims=True)) / np.where(s > 0, s, 1.0)


SCALERS = {"SNV": snv, "min-max": lambda X: np.asarray(X, float)}

data = load_bacteria(DATA_DIR, normalize=False)
AXIS = data.axis
MM_all, y_all = minmax(data.X_dev), data.y_dev      # 2018 cohort, per-spectrum min-max
MM_B, y_B = minmax(data.X_test), data.y_test        # 2019 cohort (external)

# Patient = block of 400 consecutive spectra; verified, never assumed
groups = np.arange(len(y_all)) // SPECTRA_PER_PATIENT
for g in np.unique(groups):
    assert np.unique(y_all[groups == g]).size == 1, "row order is not grouped by patient"
N_PAT = groups.max() + 1
PAT_CLASS = np.array([y_all[groups == g][0] for g in range(N_PAT)])

BAND_RANGE = (400.0, min(1800.0, float(AXIS.max())))
REF_BANDS = BandSet.equal_width(AXIS, 7, lo=BAND_RANGE[0], hi=BAND_RANGE[1], axis_name="cm-1")
REF_EDGES = [(b.lo, b.hi) for b in REF_BANDS.bands]   # where local defects are planted


def _take(pool, per_class, rng):
    taken = []
    for c in np.unique(PAT_CLASS[pool]):
        taken.extend(rng.choice(pool[PAT_CLASS[pool] == c], per_class, replace=False))
    taken = np.array(sorted(taken))
    return taken, np.setdiff1d(pool, taken)


def make_split(split_repeat=0):
    """The paper's split (fit 10 | limits 5 | select 5 | qc 5 isolates) and its defects."""
    rng = np.random.default_rng(SEED + split_repeat)
    pool = np.arange(N_PAT)
    p_qc, pool = _take(pool, 1, rng)
    p_sel, pool = _take(pool, 1, rng)
    p_lim, pool = _take(pool, 1, rng)
    rows = lambda p: np.flatnonzero(np.isin(groups, p))  # noqa: E731
    ix = {"fit": rows(pool), "lim": rows(p_lim), "sel": rows(p_sel), "qc": rows(p_qc)}
    ix["ref"] = np.sort(np.concatenate([ix["lim"], ix["sel"], ix["qc"]]))
    return {
        "repeat": split_repeat, "p_qc": p_qc, "ix": ix,
        "chart_patients": np.sort(np.concatenate([p_lim, p_sel])),
        "sel_def": degrade.plant(MM_all[ix["sel"]], AXIS, edges=REF_EDGES, random_state=SEED + 77),
        "qc_def": degrade.plant(MM_all[ix["qc"]], AXIS, edges=REF_EDGES, random_state=SEED),
    }


def binf(X, f):
    """Average f adjacent channels: a coarser detector sampling."""
    X = np.asarray(X, float)
    if f == 1:
        return X
    n = (X.shape[-1] // f) * f
    return X[..., :n].reshape(*X.shape[:-1], n // f, f).mean(-1)


def fit_model(sp, *, n_bands=N_BANDS, topology=TOPOLOGY, scale=SCALE_NAME, mode="equal",
              regime=REGIME, n_components=N_COMPONENTS, bin_factor=1):
    """Fit the band ensemble and its Triage exactly as the paper does, for one configuration."""
    S = SCALERS[scale]
    prep = lambda X: S(binf(X, bin_factor))  # noqa: E731
    ax = binf(AXIS, bin_factor)
    ix = sp["ix"]
    if mode == "equal":
        bands = BandSet.equal_width(ax, n_bands, lo=BAND_RANGE[0], hi=BAND_RANGE[1], axis_name="cm-1")
    else:
        bands = BandSet.peak_informed(ax, binf(MM_all[ix["fit"]], bin_factor), n_bands,
                                      lo=BAND_RANGE[0], hi=BAND_RANGE[1], axis_name="cm-1")
    lim = (prep(MM_all[ix["lim"]]), y_all[ix["lim"]])
    t0 = time.perf_counter()
    model = BandNegotiationClassifier(
        bands=bands, base_estimator=PLSDA(n_components=n_components), topology=topology,
        regime=regime, calibration=CALIBRATION, calibration_data=lim,
        calibration_split=CAL_SPLIT, random_state=SEED).fit(prep(MM_all[ix["fit"]]), y_all[ix["fit"]])
    t_fit = time.perf_counter() - t0
    tri = Triage(conflict_score="dg", review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,
                 n_components=n_components).fit(
        model, *lim, orientation_data=(prep(sp["sel_def"].X), sp["sel_def"].degraded))
    return {"model": model, "triage": tri, "bands": bands, "prep": prep, "t_fit": t_fit}


def localise(sp, fm, planted_intervals=None, trials=None):
    """Plant a local defect in a batch of 100 spectra of one qc isolate; does the chart name it?

    A hit is a named band that overlaps the planted interval; chance is the share of bands
    that overlap it. By default the seven published intervals, six trials each (42 trials).
    """
    prep, bands, model = fm["prep"], fm["bands"], fm["model"]
    edges, labs = [(b.lo, b.hi) for b in bands.bands], bands.labels
    rngb = np.random.default_rng(SEED + 11)
    ref = [prep(MM_all[np.sort(rngb.choice(np.flatnonzero(groups == g), BATCH_N, replace=False))])
           for g in sp["chart_patients"]]
    prof = ControlProfile.fit(model, batches=ref, k=K_SIGMA, laser_nm=LASER_NM,
                              statistics=PROFILE_STATS, random_state=SEED)
    if trials is None:
        trials = [(lo, hi, np.random.default_rng(500 * t + rep))
                  for t, (lo, hi) in enumerate(REF_EDGES) for rep in range(LOCALISE_REPEATS)]
    hits, chance = [], []
    for lo, hi, r in trials:
        sel = (AXIS >= lo) & (AXIS < hi)
        overlap = [j for j, (a, b) in enumerate(edges) if a < hi and b > lo]
        g = r.choice(sp["p_qc"])
        Xl = MM_all[np.sort(r.choice(np.flatnonzero(groups == g), BATCH_N, replace=False))].copy()
        Xl[:, sel] = Xl[:, sel] * 0.55 + r.normal(0, 0.02, (len(Xl), int(sel.sum())))
        tab = prof.check(prep(Xl), name="trial").table
        named = tab.assign(az=tab["z"].abs()).groupby("label")["az"].max().idxmax()
        hits.append(labs.index(named) in overlap)
        chance.append(len(overlap) / len(edges))
    return float(np.mean(hits)), float(np.mean(chance))


def evaluate(sp, *, loc=True, **config):
    """Every number the guide reports for one configuration on one split."""
    cfg = {"n_bands": N_BANDS, "topology": TOPOLOGY, "scale": SCALE_NAME, "mode": "equal",
           "regime": REGIME, "n_components": N_COMPONENTS, "bin_factor": 1, **config}
    fm = fit_model(sp, **cfg)
    ix, qd, prep = sp["ix"], sp["qc_def"], fm["prep"]
    y_qc = y_all[ix["qc"]]
    t0 = time.perf_counter()
    T = fm["triage"].assess(prep(qd.X), y_qc).table
    t_audit = time.perf_counter() - t0
    c = T["conflict"].to_numpy()
    fam = {k: roc_auc_score(qd.degraded[(qd.kind == k) | (qd.degraded == 0)],
                            c[(qd.kind == k) | (qd.degraded == 0)]) for k in degrade.PAPER_FAMILIES}
    row = {"repeat": sp["repeat"], **cfg,
           "channels_per_band": int(np.median([b.n_features for b in fm["bands"].bands])),
           "conflict_vs_degraded": roc_auc_score(qd.degraded, c),
           "conflict_vs_wrong": roc_auc_score(1 - T["correct"], c),
           "local": fam["local contamination"],
           "global_mean": float(np.mean([fam[k] for k in degrade.PAPER_FAMILIES if k != "local contamination"])),
           "acc_in_cohort": accuracy_score(y_all[ix["ref"]], fm["model"].predict(prep(MM_all[ix["ref"]]))),
           "acc_external": accuracy_score(y_B, fm["model"].predict(prep(MM_B))),
           "t_fit_s": fm["t_fit"], "t_audit_s": t_audit,
           "orientation_margin": fm["triage"].orientation_margin_}
    if loc:
        row["loc_hit"], row["loc_chance"] = localise(sp, fm)
    return row


In [2]:
def check(rows):
    """Compare this run with the values quoted in the guide and the response letter."""
    t = pd.DataFrame(rows, columns=["quantity", "this_run", "quoted", "tol"])
    t["ok"] = (t["this_run"] - t["quoted"]).abs() <= t["tol"]
    print(t.round(4).to_string(index=False))
    print("\nALL QUOTED VALUES REPRODUCE" if t["ok"].all() else
          f"\n{(~t['ok']).sum()} VALUES DO NOT REPRODUCE: correct the text or the run")
    return t

In [3]:
from scipy.special import softmax
from pypasi import noise_level

sp = make_split(0)
fm = fit_model(sp)
clf, tri, prep = fm["model"], fm["triage"], fm["prep"]
y_qc, MM_qc, qd = y_all[sp["ix"]["qc"]], MM_all[sp["ix"]["qc"]], sp["qc_def"]
X_clean, X_def = prep(MM_qc), prep(qd.X)
T = tri.assess(X_def, y_qc).table
print("conflict vs degraded:", round(roc_auc_score(qd.degraded, T["conflict"]), 3), "(paper 0.777)")

conflict vs degraded: 0.777 (paper 0.777)


## 1 · What each family does to the model

The same spectra before and after their defect. Three quantities, per family:

- **evidence shift**: how far the band posteriors move (mean total-variation distance);
- **disagreement change**: how much more the bands disagree with each other;
- **flips** and **accuracy cost**: what the shift does to the answer.

In [4]:
def band_post(X):
    return softmax(clf.band_logits(X), axis=2)            # (spectra, bands, classes)

def disagreement(P):                                     # each band's distance from the band mean
    return 0.5 * np.abs(P - P.mean(1, keepdims=True)).sum(2).mean(1)

P0, P1 = band_post(X_clean), band_post(X_def)
pred0, pred1 = clf.predict(X_clean), clf.predict(X_def)
c = T["conflict"].to_numpy()
rows = []
for k in degrade.PAPER_FAMILIES:
    m = qd.kind == k
    rows.append({"family": k, "n": int(m.sum()),
                 "conflict_auroc": roc_auc_score(qd.degraded[m | (qd.degraded == 0)], c[m | (qd.degraded == 0)]),
                 "evidence_shift": float((0.5 * np.abs(P1[m] - P0[m]).sum(2)).mean()),
                 "disagreement_change": float((disagreement(P1[m]) - disagreement(P0[m])).mean()),
                 "flips": float((pred0[m] != pred1[m]).mean()),
                 "accuracy_cost_pp": 100 * (accuracy_score(y_qc[m], pred0[m]) - accuracy_score(y_qc[m], pred1[m]))})
fam = pd.DataFrame(rows).sort_values("accuracy_cost_pp", ascending=False)
print(fam.round(4).to_string(index=False))
print(f"\nclean spectra: band disagreement {disagreement(P0).mean():.3f} ± {disagreement(P0).std():.3f}")

             family   n  conflict_auroc  evidence_shift  disagreement_change  flips  accuracy_cost_pp
local contamination 185          0.9237          0.2486               0.0814 0.1784           13.5135
     throughput dip 211          0.9078          0.2942               0.0895 0.0853            6.1611
    resolution loss 207          0.8041          0.2009               0.0506 0.1208            1.4493
         axis shift 205          0.7569          0.2152               0.0522 0.0585            0.9756
     short exposure 192          0.4846          0.0210               0.0001 0.0052            0.5208

clean spectra: band disagreement 0.307 ± 0.096


## 2 · How large is the planted noise against the noise already in the data?

`pypasi.noise_level` is the robust spread of the Savitzky–Golay residual, computed on the min–max spectra (before SNV).

In [5]:
native = noise_level(MM_qc)
I = MM_qc.mean()
planted_lo, planted_hi = 0.02 * 0.3 * np.sqrt(I + 0.05), 0.02 * 1.0 * np.sqrt(I + 0.05)
print(f"noise already in the spectra : median {np.median(native):.3f} of the min-max range")
print(f"planted short-exposure noise : sd {planted_lo:.4f} to {planted_hi:.4f} at the mean intensity")
print(f"-> the planted noise adds at most {100 * (planted_hi / np.median(native)) ** 2:.1f}% to the noise variance")
nz = noise_level(qd.X)
print("\nnoise_level against each of the paper's families:")
for k in degrade.PAPER_FAMILIES:
    m = (qd.kind == k) | (qd.degraded == 0)
    print(f"  {k:20s} {roc_auc_score(qd.degraded[m], nz[m]):.3f}")

noise already in the spectra : median 0.097 of the min-max range
planted short-exposure noise : sd 0.0043 to 0.0144 at the mean intensity
-> the planted noise adds at most 2.2% to the noise variance

noise_level against each of the paper's families:
  axis shift           0.110
  resolution loss      0.000
  throughput dip       0.324
  short exposure       0.507
  local contamination  0.414


## 3 · A noise ladder: when the noise does matter, who sees it?

Short exposure alone, at 1–32 times the paper's level, on half the qc spectra.

In [6]:
lad = []
for mult in (1, 2, 4, 8, 16, 32):
    r = np.random.default_rng(100 + mult)
    MMd, lab = MM_qc.copy(), np.zeros(len(MM_qc), int)
    bad = r.permutation(len(MMd))[: len(MMd) // 2]
    for i, s in zip(bad, r.uniform(0.3, 1.0, len(bad))):
        MMd[i] = MMd[i] + r.normal(0, 0.02 * mult * s * np.sqrt(np.clip(MMd[i], 0, None) + 0.05), MMd.shape[1])
        lab[i] = 1
    Xd = prep(MMd)
    Td, pd_ = tri.assess(Xd, y_qc).table, clf.predict(Xd)
    lad.append({"times_paper": mult,
                "accuracy_cost_pp": 100 * (accuracy_score(y_qc[lab == 1], pred0[lab == 1]) - accuracy_score(y_qc[lab == 1], pd_[lab == 1])),
                "conflict": roc_auc_score(lab, Td["conflict"]),
                "novelty": roc_auc_score(lab, Td["novelty"]),
                "noise_level": roc_auc_score(lab, noise_level(MMd)),
                "1 - confidence": roc_auc_score(lab, 1 - Td["confidence"]),
                "disagreement_change": float((disagreement(band_post(Xd)) - disagreement(P0))[lab == 1].mean())})
lad = pd.DataFrame(lad)
print(lad.round(3).to_string(index=False))

 times_paper  accuracy_cost_pp  conflict  novelty  noise_level  1 - confidence  disagreement_change
           1               0.0     0.524    0.526        0.524           0.494               -0.000
           2               0.1     0.497    0.499        0.498           0.505                0.000
           4               3.0     0.527    0.570        0.626           0.498                0.003
           8              10.6     0.563    0.661        0.826           0.540                0.014
          16              37.4     0.692    0.820        0.976           0.547                0.040
          32              61.1     0.876    0.944        1.000           0.327                0.050


## 4 · Every value quoted, checked

In [7]:
F = fam.set_index("family"); L = lad.set_index("times_paper")
claims = check([
    ("short exposure: flips",                 F.loc["short exposure", "flips"], 0.005, 0.002),
    ("short exposure: accuracy cost (pp)",    F.loc["short exposure", "accuracy_cost_pp"], 0.5, 0.15),
    ("local contamination: accuracy cost",    F.loc["local contamination", "accuracy_cost_pp"], 13.5, 0.15),
    ("short exposure: evidence shift",        F.loc["short exposure", "evidence_shift"], 0.021, 0.002),
    ("native noise (min-max units)",          float(np.median(native)), 0.10, 0.01),
    ("noise_level vs short exposure (x1)",    float(roc_auc_score(qd.degraded[(qd.kind == "short exposure") | (qd.degraded == 0)], nz[(qd.kind == "short exposure") | (qd.degraded == 0)])), 0.50, 0.01),
    ("x8: accuracy cost (pp)",                L.loc[8, "accuracy_cost_pp"], 10.6, 0.15),
    ("x8: conflict",                          L.loc[8, "conflict"], 0.56, 0.01),
    ("x8: novelty",                           L.loc[8, "novelty"], 0.66, 0.01),
    ("x8: noise_level",                       L.loc[8, "noise_level"], 0.83, 0.01),
    ("x8: 1 - confidence",                    L.loc[8, "1 - confidence"], 0.54, 0.01),
    ("x16: conflict",                         L.loc[16, "conflict"], 0.69, 0.01),
    ("x16: noise_level",                      L.loc[16, "noise_level"], 0.98, 0.01),
    ("planted sd at mean intensity, s = 0.3", planted_lo, 0.0043, 0.0002),
    ("planted sd at mean intensity, s = 1.0", planted_hi, 0.0144, 0.0002),
])
fam.to_csv("short_exposure_families.csv", index=False); lad.to_csv("short_exposure_ladder.csv", index=False)

                             quantity  this_run  quoted    tol   ok
                short exposure: flips    0.0052  0.0050 0.0020 True
   short exposure: accuracy cost (pp)    0.5208  0.5000 0.1500 True
   local contamination: accuracy cost   13.5135 13.5000 0.1500 True
       short exposure: evidence shift    0.0210  0.0210 0.0020 True
         native noise (min-max units)    0.0967  0.1000 0.0100 True
   noise_level vs short exposure (x1)    0.5067  0.5000 0.0100 True
               x8: accuracy cost (pp)   10.6000 10.6000 0.1500 True
                         x8: conflict    0.5632  0.5600 0.0100 True
                          x8: novelty    0.6606  0.6600 0.0100 True
                      x8: noise_level    0.8264  0.8300 0.0100 True
                   x8: 1 - confidence    0.5400  0.5400 0.0100 True
                        x16: conflict    0.6917  0.6900 0.0100 True
                     x16: noise_level    0.9764  0.9800 0.0100 True
planted sd at mean intensity, s = 0.3    0.0043 

## 5 · The three further families of `pypasi.degrade`

Cosmic rays, a fluorescence background and intensity loss, planted with `degrade.stress_test` on the same held-out spectra (Reviewer 1, comment 4). Intensity loss is removed exactly by SNV, so no spectral score can see it.

In [8]:
extra = degrade.stress_test(tri, MM_qc, y_qc, AXIS, preprocess=prep, edges=REF_EDGES,
                            families=degrade.EXTRA_FAMILIES, random_state=SEED)
print(extra.round(3).to_string(index=False))
E = extra.set_index("family")
check([
    ("fluorescence: conflict",            E.loc["fluorescence", "conflict"], 0.996, 0.005),
    ("fluorescence: accuracy cost (pp)",  E.loc["fluorescence", "accuracy_cost_pp"], 28.1, 0.15),
    ("cosmic rays: conflict",             E.loc["cosmic rays", "conflict"], 0.503, 0.005),
    ("cosmic rays: accuracy cost (pp)",   E.loc["cosmic rays", "accuracy_cost_pp"], 0.6, 0.15),
    ("intensity loss: conflict",          E.loc["intensity loss", "conflict"], 0.498, 0.005),
    ("intensity loss: accuracy cost (pp)",E.loc["intensity loss", "accuracy_cost_pp"], 0.0, 0.15),
])
extra.to_csv("degrade_extra_families.csv", index=False)

        family    n  conflict  1 - confidence  novelty  accuracy_before  accuracy_after  accuracy_cost_pp
   cosmic rays  336     0.503           0.514    0.535            0.926           0.920             0.595
  fluorescence  345     0.996           0.390    0.665            0.933           0.652            28.116
intensity loss  319     0.498           0.494    0.499            0.881           0.881             0.000
           all 1000     0.672           0.465    0.568            0.914           0.815             9.900
                          quantity  this_run  quoted   tol   ok
            fluorescence: conflict    0.9959   0.996 0.005 True
  fluorescence: accuracy cost (pp)   28.1159  28.100 0.150 True
             cosmic rays: conflict    0.5033   0.503 0.005 True
   cosmic rays: accuracy cost (pp)    0.5952   0.600 0.150 True
          intensity loss: conflict    0.4981   0.498 0.005 True
intensity loss: accuracy cost (pp)    0.0000   0.000 0.150 True

ALL QUOTED VALUES REP